# Estudo 8 — A fórmula que decide quanto você deveria pagar de dívida por mês
### @financelabr — Série "Economia em Dados" (fechamento da trilogia sobre dívidas)

Este notebook implementa o modelo apresentado no estudo: dado (a) a taxa de juros da dívida, (b) a renda líquida e (c) o total da dívida, calcula a **faixa recomendada** de comprometimento mensal — sem nunca violar o mínimo existencial.

**Lógica resumida:**
1. Preserva um piso protegido (Mínimo Existencial).
2. Calcula quanto sobra da renda além do piso (Renda Disponível).
3. Mede a "urgência" da dívida comparando sua taxa com o CDI (referência de custo de oportunidade) e com o rotativo do cartão (a modalidade mais cara do mercado).
4. Aplica essa urgência sobre a Renda Disponível, com um teto de segurança de 70% (nunca compromete 100%, pra manter colchão de emergência).


> ## 🔒 Modelo v2.2 — versão congelada para publicação
>
> | Versão | O que mudou |
> |---|---|
> | v1 | Modelo inicial — urgência normalizada pelo rotativo do cartão |
> | v2 | Separação entre capacidade de comprometimento e alocação por urgência; classificação por spread sobre o CDI líquido |
> | v2.1 | Parcela mínima contratual como piso; prazo com amortização real (juros compostos), não divisão simples |
> | **v2.2** | **Diagnóstico de não amortização, distinguindo custo elevado (Alerta A) de escala da dívida (Alerta B); indicador de cobertura de juros** |
>
> A partir daqui, qualquer mudança na lógica deve ser tratada como **v2.3**, baseada em evidência real de uso — não em continuar refinando com cenários sintéticos.

## 1. Parâmetros de mercado

Atualizar periodicamente com dados do BCB (Séries Temporais - SGS) e CDI (B3/ANBIMA). Valores abaixo são referência de agosto/2026.

In [ ]:
# --- Parâmetros de mercado (fonte: Banco Central do Brasil - SGS / B3) ---

CDI_ANUAL = 0.1415        # CDI ~14,15% a.a. (ago/2026)
ROTATIVO_ANUAL = 4.245    # Cartão rotativo ~424,5% a.a. (BCB, referência de dívida mais cara do mercado)
SALARIO_MINIMO = 1518.00  # Salário mínimo nacional vigente (atualizar todo ano)

def taxa_anual_para_mensal(taxa_anual):
    """Converte taxa anual (decimal) em taxa mensal equivalente (juros compostos)."""
    return (1 + taxa_anual) ** (1 / 12) - 1

CDI_MENSAL = taxa_anual_para_mensal(CDI_ANUAL)
ROTATIVO_MENSAL = taxa_anual_para_mensal(ROTATIVO_ANUAL)

print(f"CDI mensal:      {CDI_MENSAL*100:.2f}%")
print(f"Rotativo mensal: {ROTATIVO_MENSAL*100:.2f}%")


## 2. O modelo

In [ ]:
def calcular_minimo_existencial(renda_liquida):
    """Piso protegido: nunca abaixo do salário mínimo, e escala com a renda pra rendas maiores."""
    return max(SALARIO_MINIMO, 0.40 * renda_liquida)


def calcular_urgencia(taxa_mensal_divida):
    """
    Normaliza a taxa da dívida entre 0 (tão barata quanto o CDI, não vale antecipar)
    e 1 (tão cara quanto o rotativo do cartão, prioridade máxima).
    """
    numerador = taxa_mensal_divida - CDI_MENSAL
    denominador = ROTATIVO_MENSAL - CDI_MENSAL
    u = numerador / denominador
    return min(max(u, 0), 1)  # clamp entre 0 e 1


def calcular_faixa_recomendada(renda_liquida, taxa_anual_divida, total_divida=None, teto_seguranca=0.70):
    """
    Calcula a faixa recomendada de comprometimento mensal com dívida.

    Parâmetros
    ----------
    renda_liquida : renda líquida mensal (R$)
    taxa_anual_divida : taxa de juros anual da dívida, em decimal (ex: 1.5 = 150% a.a.)
    total_divida : total da dívida em aberto (opcional, estima prazo de quitação)
    teto_seguranca : fração máxima da Renda Disponível comprometida mesmo em urgência máxima

    Retorna
    -------
    dict com o resultado detalhado
    """
    taxa_mensal = taxa_anual_para_mensal(taxa_anual_divida)
    me = calcular_minimo_existencial(renda_liquida)
    rd = renda_liquida - me

    if rd <= 0:
        return {
            "erro": "Renda insuficiente para cobrir o mínimo existencial. Priorize renegociação/repactuação da dívida antes de qualquer comprometimento adicional.",
            "minimo_existencial": round(me, 2),
            "renda_disponivel": round(rd, 2),
        }

    u = calcular_urgencia(taxa_mensal)
    valor_min = 0.50 * u * rd
    valor_max = teto_seguranca * u * rd

    resultado = {
        "renda_liquida": round(renda_liquida, 2),
        "taxa_anual_divida_pct": round(taxa_anual_divida * 100, 1),
        "taxa_mensal_divida_pct": round(taxa_mensal * 100, 2),
        "minimo_existencial": round(me, 2),
        "renda_disponivel": round(rd, 2),
        "urgencia_0a1": round(u, 2),
        "faixa_recomendada_mes": (round(valor_min, 2), round(valor_max, 2)),
        "pct_da_renda_liquida": (
            round(100 * valor_min / renda_liquida, 1),
            round(100 * valor_max / renda_liquida, 1),
        ),
    }

    if total_divida:
        meses_min = total_divida / valor_max if valor_max > 0 else float("inf")
        meses_max = total_divida / valor_min if valor_min > 0 else float("inf")
        resultado["prazo_estimado_meses"] = (round(meses_min, 1), round(meses_max, 1))

    return resultado


## 3. Testando com as personas do estudo

In [ ]:
import pandas as pd

personas = [
    {"nome": "Renda baixa + dívida cara (rotativo cartão)", "renda": 2200, "taxa_aa": 4.245, "divida_total": 3000},
    {"nome": "Renda média + dívida moderada (crédito pessoal)", "renda": 6000, "taxa_aa": 1.066, "divida_total": 12000},
    {"nome": "Renda alta + dívida barata (financiamento veículo)", "renda": 15000, "taxa_aa": 0.27, "divida_total": 40000},
    {"nome": "Renda baixa + cheque especial (teto legal)", "renda": 2800, "taxa_aa": 1.5182, "divida_total": 2500},
]

linhas = []
for p in personas:
    r = calcular_faixa_recomendada(p["renda"], p["taxa_aa"], p["divida_total"])
    linhas.append({
        "Perfil": p["nome"],
        "Renda líquida": f"R$ {p['renda']:,.0f}",
        "Taxa da dívida (a.a.)": f"{r['taxa_anual_divida_pct']}%",
        "Mínimo existencial": f"R$ {r['minimo_existencial']:,.0f}",
        "Urgência (0-1)": r["urgencia_0a1"],
        "Faixa recomendada/mês": f"R$ {r['faixa_recomendada_mes'][0]:,.0f} – {r['faixa_recomendada_mes'][1]:,.0f}",
        "% da renda": f"{r['pct_da_renda_liquida'][0]}% – {r['pct_da_renda_liquida'][1]}%",
        "Prazo estimado (meses)": f"{r['prazo_estimado_meses'][0]:.0f} – {r['prazo_estimado_meses'][1]:.0f}",
    })

df = pd.DataFrame(linhas)
df


## 4. Gráfico — % da renda comprometida por perfil

Útil pro carrossel do Instagram / capa do LinkedIn: mostra visualmente que **não existe número fixo pra todo mundo**.

In [ ]:
import matplotlib.pyplot as plt

nomes = [p["nome"].split(" (")[0] for p in personas]
faixas = [calcular_faixa_recomendada(p["renda"], p["taxa_aa"], p["divida_total"])["pct_da_renda_liquida"] for p in personas]
minimos = [f[0] for f in faixas]
maximos = [f[1] for f in faixas]

fig, ax = plt.subplots(figsize=(9, 5))
y_pos = range(len(nomes))
ax.barh(y_pos, [mx - mn for mn, mx in zip(minimos, maximos)], left=minimos, height=0.5, color="#2E86AB")
for i, (mn, mx) in enumerate(zip(minimos, maximos)):
    ax.text(mx + 0.5, i, f"{mn}%–{mx}%", va="center", fontsize=10)

ax.set_yticks(y_pos)
ax.set_yticklabels(nomes)
ax.set_xlabel("% da renda líquida recomendada para dívida")
ax.set_title("Não existe número fixo: o % ideal muda com a taxa da dívida")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("faixa_por_perfil.png", dpi=150)
plt.show()


## 5. Gerador de resposta rápida (funil de comentários/DM)

Função pronta pra colar a renda e a taxa da dívida que a pessoa comentar e devolver o texto de resposta.

In [ ]:
def gerar_resposta_rapida(renda, taxa_anual_pct, divida_total=None):
    """
    Uso rápido pra responder comentários/DMs.
    taxa_anual_pct: em % (ex: 424.5 para o rotativo do cartão)
    """
    r = calcular_faixa_recomendada(renda, taxa_anual_pct / 100, divida_total)
    if "erro" in r:
        return f"⚠️ {r['erro']}"

    txt = (
        f"Com renda líquida de R$ {renda:,.0f} e essa dívida (~{r['taxa_anual_divida_pct']}% a.a.), "
        f"o modelo recomenda comprometer entre R$ {r['faixa_recomendada_mes'][0]:,.0f} "
        f"e R$ {r['faixa_recomendada_mes'][1]:,.0f} por mês "
        f"({r['pct_da_renda_liquida'][0]}% a {r['pct_da_renda_liquida'][1]}% da renda), "
        f"preservando R$ {r['minimo_existencial']:,.0f} como mínimo pra viver."
    )
    if "prazo_estimado_meses" in r:
        txt += (
            f" Nesse ritmo, a quitação leva entre {r['prazo_estimado_meses'][0]:.0f} "
            f"e {r['prazo_estimado_meses'][1]:.0f} meses."
        )
    return txt


# Exemplo de uso — troque pelos números de quem comentar/mandar DM:
print(gerar_resposta_rapida(renda=3500, taxa_anual_pct=181.2, divida_total=5000))


## 6. Regra de bolso simplificada (pra quem não roda código)

Versão didática pro carrossel/reel — três faixas de urgência sem precisar da fórmula completa.

In [ ]:
def regra_de_bolso(taxa_anual_pct):
    """Classifica a dívida em 3 faixas de urgência e devolve a orientação simplificada."""
    if taxa_anual_pct >= 150:
        return "🔴 Dívida cara (rotativo, cheque especial): ataque com tudo — até 70% do que sobra além do mínimo pra viver."
    elif taxa_anual_pct >= 40:
        return "🟡 Dívida moderada (crédito pessoal, parcelado): equilibre — algo entre 20% e 40% do que sobra."
    else:
        return "🟢 Dívida barata (consignado, financiamento, perto do CDI): não precisa correr — pague o mínimo contratual e deixe o resto rendendo."


for taxa in [424.5, 106.6, 27.0, 14.15]:
    print(f"{taxa:>6.1f}% a.a. -> {regra_de_bolso(taxa)}")


---
### Notas de metodologia (transparência pro conteúdo)
- **Mínimo existencial**: a Lei 14.181/2021 (Superendividamento) exige a preservação de um piso, mas não define um valor objetivo. A premissa `max(salário mínimo, 40% da renda)` é uma escolha metodológica deste estudo, não um valor legal — isso deve ficar explícito nos posts.
- **CDI e taxas de crédito**: fonte BCB (SGS) e B3/ANBIMA. Atualizar `CDI_ANUAL`, `ROTATIVO_ANUAL` e `SALARIO_MINIMO` periodicamente.
- **Teto de segurança de 70%**: evita recomendar comprometimento total do excedente, preservando colchão de emergência — sem isso, a pessoa quita a dívida e recai em uma nova no mês seguinte.


## 7. Análise de sensibilidade

Quanto a faixa recomendada muda se ajustarmos as duas premissas do modelo (multiplicador do mínimo existencial e teto de segurança)? Testa aqui antes de fechar os parâmetros pro conteúdo final.

In [ ]:
def calcular_faixa_recomendada_custom(renda_liquida, taxa_anual_divida, mult_me=0.40, teto_seguranca=0.70):
    """Mesma lógica de calcular_faixa_recomendada, mas com mult_me e teto_seguranca variáveis."""
    taxa_mensal = taxa_anual_para_mensal(taxa_anual_divida)
    me = max(SALARIO_MINIMO, mult_me * renda_liquida)
    rd = renda_liquida - me
    if rd <= 0:
        return None
    u = calcular_urgencia(taxa_mensal)
    valor_min = 0.50 * u * rd
    valor_max = teto_seguranca * u * rd
    return (round(valor_min, 2), round(valor_max, 2))


# Perfil de referência para o teste: renda média + dívida moderada
renda_teste = 6000
taxa_teste = 1.066  # 106,6% a.a. (crédito pessoal)

mult_me_opcoes = [0.30, 0.40, 0.50]
teto_opcoes = [0.60, 0.70, 0.80]

linhas_sens = []
for mult_me in mult_me_opcoes:
    for teto in teto_opcoes:
        faixa = calcular_faixa_recomendada_custom(renda_teste, taxa_teste, mult_me, teto)
        linhas_sens.append({
            "Mult. Mínimo Existencial": f"{mult_me*100:.0f}%",
            "Teto de segurança": f"{teto*100:.0f}%",
            "Faixa recomendada/mês": f"R$ {faixa[0]:,.0f} – {faixa[1]:,.0f}",
        })

df_sens = pd.DataFrame(linhas_sens)
df_sens

---
# PARTE 2 — Modelo v2 (revisado)

Revisão feita a partir de feedback técnico sobre o v1. Três problemas corrigidos:

1. **Capacidade e alocação eram a mesma conta.** Agora são duas decisões separadas: primeiro quanto a pessoa *pode* comprometer (renda − mínimo existencial − margem de segurança), depois quanto dessa capacidade *deve* ir pra dívida (urgência).
2. **Urgência normalizada pelo rotativo (424,5% a.a.) comprimia qualquer taxa moderada perto de zero.** Uma dívida de 27% a.a. — visivelmente acima do custo de oportunidade — recebia U≈0,07 e um prazo de recomendação de 8 a 11 anos. Trocado por classificação em **spread de pontos percentuais sobre o CDI líquido**, que já é internamente mais estável e não depende da pior dívida do mercado como referência.
3. **CDI nominal em vez de líquido.** Rendimento de CDI sofre IR regressivo (22,5% a 15% conforme o prazo). Usar CDI bruto superestima o retorno alternativo e torna qualquer dívida parecer mais "cara" do que realmente é em termos líquidos.

In [ ]:
IR_APROXIMADO = 0.15  # alíquota regressiva mínima (>720 dias) - premissa conservadora, documentada
CDI_LIQUIDO_ANUAL = CDI_ANUAL * (1 - IR_APROXIMADO)

print(f"CDI bruto:   {CDI_ANUAL*100:.2f}% a.a.")
print(f"CDI líquido: {CDI_LIQUIDO_ANUAL*100:.2f}% a.a.  (aproximação: IR de {IR_APROXIMADO*100:.0f}% sobre o CDI bruto)")


## Etapa 1 — Capacidade de comprometimento

Quanto a pessoa *pode* destinar a qualquer compromisso extra, sem depender do tipo de dívida:

```
Capacidade = (Renda − Mínimo Existencial) × (1 − margem de segurança)
```

A margem de segurança (padrão 15%) fica **sempre** reservada, independente da urgência — resolve a contradição do "ataque com tudo" que na prática deixava 30% de folga.

## Etapa 2 — Alocação da capacidade, por spread sobre o CDI líquido

```
spread (p.p.) = taxa anual da dívida − CDI líquido anual
```

| Spread | Classificação | % da capacidade alocado |
|---|---|---|
| ≥ 300 p.p. | 🔴 Extremamente cara | 85% – 100% |
| 100 – 300 p.p. | 🔴 Muito cara | 65% – 85% |
| 40 – 100 p.p. | 🟠 Cara | 40% – 65% |
| 5 – 40 p.p. | 🟡 Moderada | 15% – 40% |
| < 5 p.p. | 🟢 Baixa | 0% – 15% (mínimo contratual) |

Os limites são **parâmetros heurísticos do modelo**, calibrados para produzir diferentes níveis de agressividade conforme o custo relativo da dívida — não uma estimação empírica. Os exemplos do estudo (rotativo ~410 p.p., cheque especial ~138 p.p., crédito pessoal ~93 p.p., financiamento ~13 p.p., consignado ~0 p.p.) foram usados pra calibrar os limiares, não pra validá-los estatisticamente. Essa distinção importa se o método for publicado como metodologia, não só como conteúdo.

In [ ]:
TIERS = [
    (300, float("inf"), "Extremamente cara", "🔴", 0.85, 1.00),
    (100, 300,           "Muito cara",        "🔴", 0.65, 0.85),
    (40,  100,           "Cara",              "🟠", 0.40, 0.65),
    (5,   40,            "Moderada",          "🟡", 0.15, 0.40),
    (float("-inf"), 5,   "Baixa",             "🟢", 0.00, 0.15),
]

def classificar_urgencia_v2(spread_pp):
    """Classifica a dívida pelo spread (em p.p.) sobre o CDI líquido."""
    for lo, hi, label, emoji, amin, amax in TIERS:
        if lo <= spread_pp < hi:
            return label, emoji, amin, amax


def calcular_faixa_v2(renda_liquida, taxa_anual_divida, total_divida=None,
                       mult_me=0.40, margem_seguranca=0.15):
    """
    Modelo v2: separa capacidade de comprometimento (etapa 1) de alocação por
    urgência (etapa 2), usando spread sobre o CDI líquido em vez de normalização
    pelo rotativo do cartão.
    """
    me = max(SALARIO_MINIMO, mult_me * renda_liquida)
    rd = renda_liquida - me

    if rd <= 0:
        return {
            "erro": "Renda insuficiente para cobrir o mínimo existencial. Priorize renegociação/repactuação da dívida antes de qualquer comprometimento adicional.",
            "minimo_existencial": round(me, 2),
        }

    capacidade = rd * (1 - margem_seguranca)
    spread_pp = (taxa_anual_divida - CDI_LIQUIDO_ANUAL) * 100
    label, emoji, amin, amax = classificar_urgencia_v2(spread_pp)

    valor_min = amin * capacidade
    valor_max = amax * capacidade

    resultado = {
        "renda_liquida": round(renda_liquida, 2),
        "taxa_anual_divida_pct": round(taxa_anual_divida * 100, 1),
        "spread_pp": round(spread_pp, 1),
        "classificacao": f"{emoji} {label}",
        "minimo_existencial": round(me, 2),
        "renda_disponivel": round(rd, 2),
        "capacidade": round(capacidade, 2),
        "faixa_recomendada_mes": (round(valor_min, 2), round(valor_max, 2)),
        "pct_da_renda_liquida": (
            round(100 * valor_min / renda_liquida, 1),
            round(100 * valor_max / renda_liquida, 1),
        ),
    }

    if total_divida:
        meses_min = total_divida / valor_max if valor_max > 0 else float("inf")
        meses_max = total_divida / valor_min if valor_min > 0 else float("inf")
        resultado["prazo_estimado_meses"] = (round(meses_min, 1), round(meses_max, 1))

    return resultado


## Comparação direta: v1 vs v2 nas mesmas personas

Aqui fica visível o que mudou — principalmente no caso de R$15.000, que era o mais problemático.

In [ ]:
linhas_cmp = []
for p in personas:
    r1 = calcular_faixa_recomendada(p["renda"], p["taxa_aa"], p["divida_total"])
    r2 = calcular_faixa_v2(p["renda"], p["taxa_aa"], p["divida_total"])
    linhas_cmp.append({
        "Perfil": p["nome"],
        "v1 - Faixa/mês": f"R$ {r1['faixa_recomendada_mes'][0]:,.0f} – {r1['faixa_recomendada_mes'][1]:,.0f}",
        "v1 - Prazo (m)": f"{r1['prazo_estimado_meses'][0]:.0f} – {r1['prazo_estimado_meses'][1]:.0f}",
        "v2 - Classificação": r2["classificacao"],
        "v2 - Faixa/mês": f"R$ {r2['faixa_recomendada_mes'][0]:,.0f} – {r2['faixa_recomendada_mes'][1]:,.0f}",
        "v2 - Prazo (m)": f"{r2['prazo_estimado_meses'][0]:.0f} – {r2['prazo_estimado_meses'][1]:.0f}",
    })

df_cmp = pd.DataFrame(linhas_cmp)
df_cmp


## Gerador de resposta rápida — v2

Reescrito também na frase: nada de "ataque com tudo" contradizendo um teto de 70%. O texto agora reflete exatamente o que a fórmula calcula.

In [ ]:
def gerar_resposta_rapida_v2(renda, taxa_anual_pct, divida_total=None):
    r = calcular_faixa_v2(renda, taxa_anual_pct / 100, divida_total)
    if "erro" in r:
        return f"⚠️ {r['erro']}"

    txt = (
        f"Com renda líquida de R$ {renda:,.0f} e essa dívida a {r['taxa_anual_divida_pct']}% a.a. "
        f"({r['classificacao']}, spread de {r['spread_pp']:.0f} p.p. sobre o CDI líquido), "
        f"o modelo recomenda comprometer entre R$ {r['faixa_recomendada_mes'][0]:,.0f} "
        f"e R$ {r['faixa_recomendada_mes'][1]:,.0f} por mês "
        f"({r['pct_da_renda_liquida'][0]}% a {r['pct_da_renda_liquida'][1]}% da renda), "
        f"preservando R$ {r['minimo_existencial']:,.0f} como mínimo pra viver."
    )
    if "prazo_estimado_meses" in r:
        txt += (
            f" Nesse ritmo, a quitação leva entre {r['prazo_estimado_meses'][0]:.0f} "
            f"e {r['prazo_estimado_meses'][1]:.0f} meses."
        )
    return txt


print(gerar_resposta_rapida_v2(renda=3500, taxa_anual_pct=181.2, divida_total=5000))
print()
print(gerar_resposta_rapida_v2(renda=15000, taxa_anual_pct=27.0, divida_total=40000))


### Nota de metodologia atualizada
- **CDI líquido**: aproximação usando a alíquota mínima de IR regressivo (15%, válida para resgates acima de 720 dias). É uma simplificação didática — o IR real depende do prazo de cada aplicação — mas evita o erro de comparar dívida com CDI bruto.
- **Classificação por spread (p.p.), não por taxa absoluta**: "27% a.a. não é dívida barata só por estar longe do rotativo" — agora a régua é o quanto a dívida excede o retorno líquido disponível, não a distância até o pior caso do mercado.
- **Margem de segurança separada da urgência**: a reserva de 15% (ou o que for definido) nunca é comprometida, mesmo em dívida extremamente cara — elimina a contradição entre "até 70%" e "ataque com tudo".

---
# PARTE 3 — Modelo v2.1 (parcela mínima + amortização real)

Três correções sobre o v2:

1. **Renomeado "margem de segurança" → "margem de proteção mensal".** Ela não forma reserva de emergência — é só um buffer que fica de fora do cálculo de alocação. O nome antigo sugeria uma função que o modelo não cumpre.
2. **Nova entrada: parcela/mínimo contratual.** Sem isso, o modelo não tinha como garantir que a recomendação nunca fique abaixo da obrigação já assumida. Agora: `pagamento = max(parcela mínima, alocação recomendada)` — com alerta se a própria parcela mínima já estourar a capacidade segura.
3. **Prazo corrigido para amortização real com juros compostos**, em vez de divisão simples (dívida ÷ pagamento). Essa era a correção mais importante: pra uma dívida de R$50.000 a 27% a.a. pagando R$1.148/mês, a divisão simples dizia 43,6 meses — a conta real, com juros incidindo sobre o saldo, dá **104,9 meses**. Mais que o dobro.

In [ ]:
import math

def prazo_amortizacao(divida, taxa_mensal, pagamento):
    """
    Prazo real de quitação (em meses), considerando juros compostos sobre o saldo devedor.
    Resolve n a partir de: D = P * (1 - (1+i)^-n) / i
    Retorna None se o pagamento não cobre nem os juros do período (dívida nunca é quitada nesse ritmo).
    """
    if divida is None or divida <= 0:
        return None
    juros_periodo = divida * taxa_mensal
    if pagamento <= juros_periodo:
        return None
    n = -math.log(1 - taxa_mensal * divida / pagamento) / math.log(1 + taxa_mensal)
    return n


In [ ]:
def calcular_v21(renda_liquida, taxa_anual_divida, total_divida=None, parcela_minima=None,
                  mult_me=0.40, margem_protecao=0.15):
    """
    Modelo v2.1: adiciona parcela mínima contratual como piso de pagamento e calcula
    o prazo de quitação com amortização real (juros compostos sobre o saldo), em vez
    de divisão simples.
    """
    me = max(SALARIO_MINIMO, mult_me * renda_liquida)
    excedente = renda_liquida - me
    if excedente <= 0:
        return {"erro": "Renda insuficiente para cobrir o mínimo existencial."}

    capacidade = excedente * (1 - margem_protecao)
    taxa_mensal = taxa_anual_para_mensal(taxa_anual_divida)
    spread_pp = (taxa_anual_divida - CDI_LIQUIDO_ANUAL) * 100
    label, emoji, amin, amax, texto = classificar_urgencia_v21(spread_pp)

    aloc_min, aloc_max = amin * capacidade, amax * capacidade

    alerta = None
    if parcela_minima is not None:
        if parcela_minima > capacidade:
            alerta = (
                f"Sua parcela mínima (R$ {parcela_minima:,.0f}) já ultrapassa a capacidade segura "
                f"(R$ {capacidade:,.0f}) pra essa renda. O problema não é quanto pagar — é renegociar "
                f"a dívida ou reduzir despesas antes de qualquer outra decisão."
            )
            pagamento_min = pagamento_max = parcela_minima
        else:
            pagamento_min = max(parcela_minima, aloc_min)
            pagamento_max = max(parcela_minima, aloc_max)
    else:
        pagamento_min, pagamento_max = aloc_min, aloc_max

    resultado = {
        "renda_liquida": round(renda_liquida, 2),
        "taxa_anual_divida_pct": round(taxa_anual_divida * 100, 1),
        "spread_pp": round(spread_pp, 1),
        "classificacao": f"{emoji} {label}",
        "orientacao": texto,
        "minimo_existencial": round(me, 2),
        "capacidade": round(capacidade, 2),
        "pagamento_recomendado": (round(pagamento_min, 2), round(pagamento_max, 2)),
        "alerta": alerta,
    }

    if total_divida:
        # pagamento maior -> prazo menor (usa o teto da faixa) e vice-versa
        p_min_meses = prazo_amortizacao(total_divida, taxa_mensal, pagamento_max)
        p_max_meses = prazo_amortizacao(total_divida, taxa_mensal, pagamento_min)
        resultado["prazo_meses"] = (
            round(p_min_meses, 1) if p_min_meses else "não amortiza nesse ritmo",
            round(p_max_meses, 1) if p_max_meses else "não amortiza nesse ritmo",
        )

    return resultado


TIERS_V21 = [
    (300, float("inf"), "Extremamente cara", "🔴", 0.85, 1.00,
     "Prioridade máxima: direcione a maior parte da sua capacidade segura para essa dívida."),
    (100, 300, "Muito cara", "🔴", 0.65, 0.85,
     "Prioridade alta: essa dívida está corroendo sua renda bem acima do que qualquer alternativa renderia."),
    (40, 100, "Cara", "🟠", 0.40, 0.65,
     "Priorize, mas com equilíbrio: reserve uma parte relevante da capacidade pra ela."),
    (5, 40, "Moderada", "🟡", 0.15, 0.40,
     "Sem pressa excessiva: uma parte pequena a moderada da capacidade já ajuda."),
    (float("-inf"), 5, "Baixa", "🟢", 0.00, 0.15,
     "Pague o mínimo contratual — antecipar tem pouco ganho real sobre deixar o dinheiro rendendo líquido."),
]

def classificar_urgencia_v21(spread_pp):
    for lo, hi, label, emoji, amin, amax, texto in TIERS_V21:
        if lo <= spread_pp < hi:
            return label, emoji, amin, amax, texto


## O caso do R$15.000, revisitado (dessa vez com o prazo correto)

In [ ]:
print("Sem parcela mínima informada:")
r = calcular_v21(15000, 0.27, total_divida=50000)
for k, v in r.items():
    print(f"  {k}: {v}")

print("\nCom parcela mínima de R$1.200 informada:")
r = calcular_v21(15000, 0.27, total_divida=50000, parcela_minima=1200)
for k, v in r.items():
    print(f"  {k}: {v}")


## Testando o alerta de incapacidade

Quando a própria parcela mínima já ultrapassa a capacidade segura — o modelo não deve simplesmente empurrar o pagamento pra cima.

In [ ]:
r = calcular_v21(2200, 4.245, total_divida=3000, parcela_minima=600)
for k, v in r.items():
    print(f"{k}: {v}")


### Nota de metodologia — v2.1
- **Prazo com amortização real**: resolve `n` a partir de `D = P × (1 − (1+i)⁻ⁿ) / i`. Se o pagamento não cobre nem os juros do período, o prazo retorna "não amortiza nesse ritmo" em vez de um número enganoso.
- **Parcela mínima como piso**: `pagamento = max(parcela mínima, alocação recomendada)`. Se a parcela mínima já ultrapassa a capacidade segura, o modelo não empurra o pagamento pra cima — devolve um alerta pra renegociação, porque nesse caso o problema não é "quanto pagar".
- **"Margem de proteção mensal"** substitui "margem de segurança" em toda a documentação — o parâmetro não constitui reserva de emergência, apenas um buffer fora do cálculo de alocação.

---
# PARTE 4 — Flexibilidade de taxa, posicionamento e teste de estresse

Últimos ajustes antes de considerar a v2.1 uma base congelada:

1. **Taxa em % a.a. ou % a.m.** — a taxa "anunciada" nem sempre é a que o usuário vai digitar. Adicionada a opção de informar em qualquer uma das duas unidades, evitando conversão errada por parte de quem for usar a calculadora.
2. **"Pagamento recomendado" ≠ "melhor pagamento possível".** O modelo não otimiza velocidade de quitação — ele aplica uma política de proteção financeira. Isso precisa estar explícito, porque alguém com capacidade de R$7.650 poderia pagar muito mais que o teto recomendado e quitar mais rápido; o modelo está deliberadamente escolhendo não recomendar isso.
3. **Teste de estresse** em centenas de cenários simulados, incluindo o teste específico de continuidade nas fronteiras entre tiers (5, 40, 100, 300 p.p.) — que é onde um modelo em degraus costuma ter problema.

In [ ]:
def taxa_para_anual(valor, unidade="aa"):
    """
    Converte a taxa informada para taxa anual efetiva (decimal), aceitando:
      unidade="aa" -> valor já é taxa anual efetiva (decimal, ex: 0.27 = 27% a.a.)
      unidade="am" -> valor é taxa mensal efetiva (decimal, ex: 0.02 = 2% a.m.)
    """
    if unidade == "aa":
        return valor
    elif unidade == "am":
        return (1 + valor) ** 12 - 1
    else:
        raise ValueError("unidade deve ser 'aa' ou 'am'")


def calcular_v21b(renda_liquida, taxa, unidade_taxa="aa", total_divida=None, parcela_minima=None,
                   mult_me=0.40, margem_protecao=0.15):
    """
    Mesma lógica da v2.1, mas aceita a taxa em % a.a. ou % a.m. (parâmetro unidade_taxa).
    Evita o erro comum de tratar uma taxa mensal como se fosse anual, ou vice-versa.
    """
    taxa_anual_divida = taxa_para_anual(taxa, unidade_taxa)
    return calcular_v21(renda_liquida, taxa_anual_divida, total_divida, parcela_minima,
                         mult_me, margem_protecao)


# Exemplo: mesma dívida informada das duas formas deve dar o mesmo resultado
r_aa = calcular_v21b(6000, 1.066, unidade_taxa="aa", total_divida=12000)
taxa_mensal_equiv = taxa_anual_para_mensal(1.066)
r_am = calcular_v21b(6000, taxa_mensal_equiv, unidade_taxa="am", total_divida=12000)

print(f"Informando 106,6% a.a.:        {r_aa['pagamento_recomendado']}")
print(f"Informando {taxa_mensal_equiv*100:.2f}% a.m. (equivalente): {r_am['pagamento_recomendado']}")
print(f"Batem? {r_aa['pagamento_recomendado'] == r_am['pagamento_recomendado']}")


## Teste 1 — Continuidade nas fronteiras dos tiers

Um modelo em degraus (5 faixas fixas) tem um risco conhecido: um centavo de diferença no spread pode saltar a recomendação inteira de uma faixa pra outra. Testando exatamente isso nos 4 limiares (5, 40, 100, 300 p.p.), pra um perfil fixo.

In [ ]:
renda_teste = 6000
capacidade_teste = (renda_teste - max(SALARIO_MINIMO, 0.40*renda_teste)) * (1-0.15)

limiares = [5, 40, 100, 300]
linhas_cont = []
for lim in limiares:
    spread_abaixo = lim - 0.01
    spread_acima = lim + 0.01
    _, _, amin_a, amax_a, _ = classificar_urgencia_v21(spread_abaixo)
    _, _, amin_d, amax_d, _ = classificar_urgencia_v21(spread_acima)
    linhas_cont.append({
        "Limiar (p.p.)": lim,
        "Faixa logo abaixo": f"{amin_a*100:.0f}%-{amax_a*100:.0f}%",
        "Faixa logo acima": f"{amin_d*100:.0f}%-{amax_d*100:.0f}%",
        "Salto no mínimo (R$)": round((amin_d-amin_a)*capacidade_teste, 2),
        "Salto no máximo (R$)": round((amax_d-amax_a)*capacidade_teste, 2),
    })

df_continuidade = pd.DataFrame(linhas_cont)
df_continuidade


**Resultado**: nos 4 limiares, o piso da faixa de cima (`amin`) é numericamente igual ao teto da faixa de baixo (`amax`) — por desenho: 15%=15%, 40%=40%, 65%=65%, 85%=85%. Isso significa que **o teto da recomendação é contínuo** na fronteira (o valor máximo não pula). Mas **o piso da recomendação salta** — porque cada tier tem seu próprio mínimo, e o mínimo da faixa de cima é sempre maior que o mínimo da faixa de baixo. Ou seja: passar de spread=39,99 pra spread=40,01 não muda o teto, mas eleva o piso da recomendação de uma vez. Pra um conteúdo público isso é aceitável (o modelo nunca fica "pior" ao cruzar a fronteira, só "mais exigente"), mas vale documentar — é uma escolha de desenho, não um bug.

## Teste 2 — Simulação em massa (Monte Carlo)

Gera cenários aleatórios variando renda, taxa e dívida, e verifica o comportamento agregado do modelo.

In [ ]:
import random

random.seed(42)
N = 2000
cenarios = []

for _ in range(N):
    renda = random.uniform(1200, 30000)
    taxa_aa = random.uniform(0.05, 4.5)  # 5% a 450% a.a.
    divida = random.uniform(500, 150000)
    tem_parcela_minima = random.random() < 0.5
    if tem_parcela_minima:
        # parcela mínima como fração aleatória (às vezes exagerada) de uma amortização "razoável"
        parcela_minima = divida * random.uniform(0.01, 0.15)
    else:
        parcela_minima = None

    r = calcular_v21(renda, taxa_aa, total_divida=divida, parcela_minima=parcela_minima)
    r["renda_input"] = renda
    r["taxa_aa_input"] = taxa_aa
    r["divida_input"] = divida
    r["parcela_minima_input"] = parcela_minima
    cenarios.append(r)

# --- Métricas agregadas ---
n_erro = sum(1 for r in cenarios if "erro" in r)
n_alerta = sum(1 for r in cenarios if r.get("alerta"))
n_nao_amortiza = sum(
    1 for r in cenarios
    if "prazo_meses" in r and ("não amortiza nesse ritmo" in r["prazo_meses"])
)

pct_renda_comprometida = [
    100 * r["pagamento_recomendado"][1] / r["renda_input"]
    for r in cenarios if "erro" not in r
]

print(f"Cenários simulados: {N}")
print(f"Renda insuficiente pro mínimo existencial: {n_erro} ({100*n_erro/N:.1f}%)")
print(f"Alertas de parcela mínima > capacidade:    {n_alerta} ({100*n_alerta/N:.1f}%)")
print(f"'Não amortiza nesse ritmo' (pelo menos uma ponta da faixa): {n_nao_amortiza} ({100*n_nao_amortiza/N:.1f}%)")
print()
print(f"% da renda comprometida (teto da faixa) — min: {min(pct_renda_comprometida):.1f}%, "
      f"mediana: {sorted(pct_renda_comprometida)[len(pct_renda_comprometida)//2]:.1f}%, "
      f"max: {max(pct_renda_comprometida):.1f}%")


In [ ]:
# Distribuição de classificação por tier
from collections import Counter

tiers_contagem = Counter(r["classificacao"] for r in cenarios if "erro" not in r)
for tier, contagem in sorted(tiers_contagem.items(), key=lambda x: -x[1]):
    print(f"{tier:25s}: {contagem:4d} cenários ({100*contagem/N:.1f}%)")


## Teste 3 — Onde o prazo "explode"

Casos em que o prazo estimado passa de 10 anos (120 meses) mesmo com pagamento acima do mínimo contratual — sinal de que a dívida está mal endereçada pela recomendação, e não só pela taxa em si.

In [ ]:
explosivos = []
for r in cenarios:
    if "prazo_meses" not in r:
        continue
    pmin, pmax = r["prazo_meses"]
    if isinstance(pmax, (int, float)) and pmax > 120:
        explosivos.append(r)

print(f"Cenários com prazo do teto da faixa acima de 120 meses: {len(explosivos)} ({100*len(explosivos)/N:.1f}%)")
if explosivos:
    exemplo = sorted(explosivos, key=lambda r: r["prazo_meses"][1])[0]
    print("\nExemplo mais próximo do limite:")
    for k in ["renda_input", "taxa_aa_input", "divida_input", "classificacao", "pagamento_recomendado", "prazo_meses"]:
        print(f"  {k}: {exemplo[k]}")


### Nota final de posicionamento
O modelo responde a uma pergunta específica — **"quanto faz sentido comprometer mensalmente, dado um piso protegido, uma margem de proteção e o custo relativo da dívida"** — e não a "qual o pagamento que quita mais rápido" nem "quanto dá pra pagar no limite". A recomendação é uma política de proteção financeira, não uma otimização de velocidade de quitação. Isso deve ficar explícito em qualquer texto publicado, principalmente porque o teste de prazo explosivo (acima) mostra situações onde alguém com capacidade de sobra teria matematicamente como quitar mais rápido pagando além do que o modelo recomenda — e essa é uma escolha normativa do modelo, não uma limitação escondida.

## Teste 4 — Investigando os dois números que saíram estranhos da simulação

Dois resultados da simulação merecem investigação antes de qualquer publicação:
1. **% da renda comprometida chegou a 1318% no pior caso** — isso é matematicamente impossível de sustentar na prática.
2. **52% dos cenários caíram em "não amortiza nesse ritmo"** — precisa separar quantos são "só o piso da faixa não é suficiente" (esperado, o piso é deliberadamente conservador) de quantos são "nem o teto da faixa resolve" (isso sim seria um problema real do modelo).

In [ ]:
# --- Investigação 1: de onde vem o 1318% de renda comprometida? ---
piores = sorted(
    [r for r in cenarios if "erro" not in r],
    key=lambda r: r["pagamento_recomendado"][1] / r["renda_input"],
    reverse=True
)[:3]

print("Piores 3 casos de % de renda comprometida:")
for r in piores:
    pct = 100 * r["pagamento_recomendado"][1] / r["renda_input"]
    print(f"  renda=R${r['renda_input']:.0f} | dívida=R${r['divida_input']:.0f} | "
          f"parcela_mínima_input={r['parcela_minima_input']} | "
          f"capacidade=R${r['capacidade']:.0f} | pagamento={r['pagamento_recomendado']} | "
          f"% renda={pct:.0f}% | alerta={'SIM' if r['alerta'] else 'não'}")


**Diagnóstico**: os casos extremos são todos cenários com `alerta` ativo — a parcela mínima informada (gerada aleatoriamente na simulação, sem relação com a renda) já é, sozinha, maior que a renda inteira. O modelo está apenas **reportando a obrigação contratual como está**, não recomendando esse valor como pagamento saudável — daí o alerta.

**O problema real não é o cálculo, é a interface**: se `pagamento_recomendado` for exibido isoladamente (por exemplo, direto num post ou resposta de DM) sem checar `alerta` primeiro, a pessoa vê "pague R$ 22.000" sem contexto. Correção: a função de resposta pro funil de comentários precisa **checar o alerta antes de tudo** e nunca apresentar o valor de pagamento como recomendação quando ele existe.

In [ ]:
def gerar_resposta_rapida_v21(renda, taxa, unidade_taxa="aa", total_divida=None, parcela_minima=None):
    """
    Versão do gerador de resposta que respeita o alerta de incapacidade:
    nunca apresenta um pagamento como 'recomendação saudável' se ele só existe
    porque a parcela mínima contratual já é maior que a capacidade segura.
    """
    taxa_aa = taxa_para_anual(taxa, unidade_taxa)
    r = calcular_v21(renda, taxa_aa, total_divida, parcela_minima)

    if "erro" in r:
        return f"⚠️ {r['erro']}"

    if r["alerta"]:
        return f"⚠️ {r['alerta']}"

    txt = (
        f"Com renda líquida de R$ {renda:,.0f} e essa dívida a {r['taxa_anual_divida_pct']}% a.a. "
        f"({r['classificacao']}, spread de {r['spread_pp']:.0f} p.p. sobre o CDI líquido), "
        f"o modelo recomenda comprometer entre R$ {r['pagamento_recomendado'][0]:,.0f} "
        f"e R$ {r['pagamento_recomendado'][1]:,.0f} por mês, "
        f"preservando R$ {r['minimo_existencial']:,.0f} como mínimo pra viver."
    )
    if "prazo_meses" in r:
        pmin, pmax = r["prazo_meses"]
        if isinstance(pmin, str) or isinstance(pmax, str):
            txt += " Nesse ritmo, o pagamento mais baixo da faixa não é suficiente pra sequer cobrir os juros — vale mirar no teto da faixa."
        else:
            txt += f" Nesse ritmo, a quitação leva entre {pmin:.0f} e {pmax:.0f} meses."
    return txt


# Reproduzindo o pior caso da simulação com a função corrigida:
pior = piores[0]
print(gerar_resposta_rapida_v21(
    pior["renda_input"], pior["taxa_aa_input"], "aa",
    pior["divida_input"], pior["parcela_minima_input"]
))


In [ ]:
# --- Investigação 2: quantos casos são "só o piso não amortiza" vs "nem o teto amortiza"? ---
so_piso_falha = 0
nem_teto_amortiza = 0

for r in cenarios:
    if "prazo_meses" not in r:
        continue
    pmin, pmax = r["prazo_meses"]  # pmin usa pagamento_max (mais rápido); pmax usa pagamento_min (mais lento)
    teto_falha = isinstance(pmin, str)
    piso_falha = isinstance(pmax, str)
    if piso_falha and not teto_falha:
        so_piso_falha += 1
    elif teto_falha:
        nem_teto_amortiza += 1

print(f"Só o piso da faixa não amortiza (esperado, é o extremo conservador): {so_piso_falha} ({100*so_piso_falha/N:.1f}%)")
print(f"Nem o teto da faixa amortiza (problema real):                        {nem_teto_amortiza} ({100*nem_teto_amortiza/N:.1f}%)")


In [ ]:
# --- % de renda comprometida, EXCLUINDO cenários com alerta ativo (pra ter uma leitura sã) ---
pct_sem_alerta = [
    100 * r["pagamento_recomendado"][1] / r["renda_input"]
    for r in cenarios if "erro" not in r and not r["alerta"]
]

print(f"Cenários sem alerta: {len(pct_sem_alerta)} de {N}")
print(f"% da renda comprometida (teto da faixa), sem os casos de alerta:")
print(f"  min: {min(pct_sem_alerta):.1f}%, mediana: {sorted(pct_sem_alerta)[len(pct_sem_alerta)//2]:.1f}%, "
      f"p95: {sorted(pct_sem_alerta)[int(0.95*len(pct_sem_alerta))]:.1f}%, max: {max(pct_sem_alerta):.1f}%")


### Conclusão do teste de estresse
- **O "1318%" não era um erro de fórmula** — era a interface reportando uma parcela contratual absurda sem checar o alerta primeiro. Corrigido com `gerar_resposta_rapida_v21`, que agora recusa apresentar pagamento quando o alerta está ativo.
- **O "52% não amortiza" precisa ser lido junto com a quebra entre piso e teto** (ver números acima) — se a maior parte for "só o piso falha", é comportamento esperado (o piso da faixa é deliberadamente o cenário mais conservador). Se uma fração relevante for "nem o teto amortiza", isso é sinal de que, pra dívidas com taxa muito alta e capacidade pequena, o modelo pode estar recomendando um valor insuficiente mesmo no limite superior — vale revisar o teto do tier 🔴 Extremamente cara nesse cenário específico.
- **Antes de publicar**: qualquer texto ou calculadora pública precisa checar `alerta` antes de exibir `pagamento_recomendado` — nunca os dois ao mesmo tempo como se fossem a mesma coisa.

## Teste 5 — Caracterizando os 42% onde nem o teto da faixa amortiza

Esse número é grande demais pra ignorar. Antes de mudar qualquer parâmetro, é preciso entender o perfil desses cenários: são casos de dívida genuinamente insustentável (o modelo está certo em não conseguir "resolver" matematicamente), ou o teto dos tiers está subdimensionado?

In [ ]:
import statistics

nao_amortiza_teto = []
for r in cenarios:
    if "prazo_meses" not in r:
        continue
    pmin, pmax = r["prazo_meses"]
    if isinstance(pmin, str):  # nem o teto (pagamento_max) amortiza
        nao_amortiza_teto.append(r)

taxas = [r["taxa_aa_input"]*100 for r in nao_amortiza_teto]
rendas = [r["renda_input"] for r in nao_amortiza_teto]
razao_divida_renda = [r["divida_input"]/r["renda_input"] for r in nao_amortiza_teto]

print(f"N = {len(nao_amortiza_teto)}")
print(f"Taxa a.a.: mediana={statistics.median(taxas):.0f}%, min={min(taxas):.0f}%, max={max(taxas):.0f}%")
print(f"Razão dívida/renda: mediana={statistics.median(razao_divida_renda):.1f}x, "
      f"min={min(razao_divida_renda):.1f}x, max={max(razao_divida_renda):.1f}x")

# Quantos desses já tinham alerta de parcela mínima também?
com_alerta = sum(1 for r in nao_amortiza_teto if r["alerta"])
print(f"Desses, já tinham alerta de parcela mínima > capacidade: {com_alerta} ({100*com_alerta/len(nao_amortiza_teto):.0f}%)")

# Olhando só os SEM alerta (esses são os que realmente importam - a recomendação normal falhando)
sem_alerta = [r for r in nao_amortiza_teto if not r["alerta"]]
print(f"\nSEM alerta prévio (a recomendação 'normal' falhou sozinha): {len(sem_alerta)} ({100*len(sem_alerta)/N:.1f}% do total)")
if sem_alerta:
    taxas2 = [r["taxa_aa_input"]*100 for r in sem_alerta]
    razao2 = [r["divida_input"]/r["renda_input"] for r in sem_alerta]
    print(f"  Taxa a.a. mediana: {statistics.median(taxas2):.0f}%")
    print(f"  Razão dívida/renda mediana: {statistics.median(razao2):.1f}x")
    exemplo = sorted(sem_alerta, key=lambda r: r["taxa_aa_input"])[0]
    print(f"  Exemplo (menor taxa do grupo): renda=R${exemplo['renda_input']:.0f}, "
          f"dívida=R${exemplo['divida_input']:.0f}, taxa={exemplo['taxa_aa_input']*100:.0f}% a.a., "
          f"capacidade=R${exemplo['capacidade']:.0f}, pagamento_max={exemplo['pagamento_recomendado'][1]:.0f}")


**Diagnóstico**: a maioria esmagadora dos "nem o teto amortiza" já tinha `alerta` de parcela mínima acionado — ou seja, são o mesmo grupo do Teste 4, casos de sobre-endividamento genuíno simulado (dívida muito acima da renda, típica do público-alvo da Lei 14.181/2021). O grupo residual, sem alerta prévio, é pequeno e concentra taxas muito altas (rotativo/cheque especial) combinadas com dívida muito maior que a renda — cenário em que mesmo 100% da capacidade não é suficiente. **Isso não é um bug do modelo: é o modelo identificando corretamente que a dívida está estruturalmente insustentável naquele nível de renda.** Só faltava um alerta dedicado pra esse caso, porque hoje ele aparece como "não amortiza" sem explicação.

In [ ]:
def calcular_v22(renda_liquida, taxa, unidade_taxa="aa", total_divida=None, parcela_minima=None,
                  mult_me=0.40, margem_protecao=0.15):
    """
    v2.2: adiciona um segundo tipo de alerta -- quando mesmo o teto da faixa recomendada
    não cobre os juros da dívida (situação de sobre-endividamento estrutural, não resolvível
    só com prioridade de pagamento).
    """
    taxa_aa = taxa_para_anual(taxa, unidade_taxa)
    r = calcular_v21(renda_liquida, taxa_aa, total_divida, parcela_minima, mult_me, margem_protecao)

    if "erro" in r or r.get("alerta"):
        return r  # já coberto pelos alertas anteriores

    if total_divida and "prazo_meses" in r:
        pmin, _ = r["prazo_meses"]
        if isinstance(pmin, str):
            r["alerta"] = (
                "Mesmo direcionando o teto da faixa recomendada, o pagamento não cobre os juros "
                "dessa dívida — ela está estruturalmente acima da capacidade de pagamento pra essa "
                "renda. Esse é um cenário de superendividamento (Lei 14.181/2021): a saída não é só "
                "priorizar o pagamento, é buscar renegociação, repactuação judicial ou redução da dívida."
            )
    return r


# Reaplicando nos cenários "sem alerta" que falharam no teto:
if sem_alerta:
    exemplo = sem_alerta[0]
    r = calcular_v22(exemplo["renda_input"], exemplo["taxa_aa_input"], "aa",
                      exemplo["divida_input"], exemplo["parcela_minima_input"])
    print(r["alerta"])


### Conclusão final do teste de estresse
- O 42,1% inicial assustava, mas na prática **quase todo esse grupo já era sobre-endividamento capturado (ou deveria ser) por um alerta** — não uma falha silenciosa da fórmula.
- **v2.2** fecha essa lacuna: agora existem dois tipos de alerta claramente distintos — (1) parcela mínima contratual maior que a capacidade segura, e (2) mesmo o teto da recomendação não sendo suficiente pra cobrir os juros. Os dois apontam pra fora do escopo de "quanto pagar" e pra dentro do escopo de "renegociar", o que conecta bem de volta com o Estudo 6 e a Lei do Superendividamento — fechando a trilogia de forma coerente.
- **Não mexeria mais na arquitetura.** A partir daqui, qualquer ajuste é calibração de parâmetro (mult_me, margem_protecao, limiares de spread), não mudança de lógica.

---
# PARTE 5 — v2.2: distinguindo causa (custo vs. escala) e cobertura de juros

O mesmo sintoma — "o pagamento recomendado não cobre os juros" — pode ter duas causas diferentes, e o texto do alerta precisa dizer qual:

- **Alerta A (custo elevado)**: a dívida está classificada como 🟠 Cara, 🔴 Muito cara ou 🔴 Extremamente cara — o problema é a taxa em si, muito acima do custo de oportunidade.
- **Alerta B (escala da dívida)**: a dívida está classificada como 🟡 Moderada ou 🟢 Baixa — a taxa não é o vilão, mas o saldo é grande demais em relação à renda pra qualquer alocação razoável dar conta.

E, em vez de só dizer "não cobre os juros", o modelo agora informa **quanto por cento dos juros o pagamento cobre** — transforma um alerta técnico em algo que a pessoa entende de imediato.

In [ ]:
def calcular_v22(renda_liquida, taxa, unidade_taxa="aa", total_divida=None, parcela_minima=None,
                  mult_me=0.40, margem_protecao=0.15):
    """
    v2.2 (congelada): mesma base da v2.1, com duas adições:
    1. Cobertura de juros: quanto % dos juros mensais o pagamento recomendado cobre.
    2. Quando o pagamento não cobre os juros, distingue a causa:
       - Alerta A (custo elevado): dívida classificada como Cara ou pior (spread >= 40 p.p.)
       - Alerta B (escala): dívida classificada como Moderada ou Baixa (spread < 40 p.p.),
         mas o saldo é grande demais pra renda mesmo com taxa razoável.
    """
    taxa_aa = taxa_para_anual(taxa, unidade_taxa)
    r = calcular_v21(renda_liquida, taxa_aa, total_divida, parcela_minima, mult_me, margem_protecao)

    if "erro" in r:
        return r

    # Parcela mínima já ultrapassando a capacidade continua sendo o alerta mais prioritário
    if r.get("alerta"):
        r["cobertura_juros_pct"] = None
        return r

    if total_divida:
        taxa_mensal = taxa_anual_para_mensal(taxa_aa)
        juros_mes = total_divida * taxa_mensal
        pagamento_max = r["pagamento_recomendado"][1]
        cobertura = round(100 * pagamento_max / juros_mes, 1) if juros_mes > 0 else None
        r["cobertura_juros_pct"] = cobertura

        if cobertura is not None and cobertura < 100:
            spread_pp = r["spread_pp"]
            if spread_pp >= 40:  # Cara, Muito cara ou Extremamente cara
                r["alerta"] = (
                    f"O pagamento recomendado cobre apenas {cobertura:.0f}% dos juros estimados do mês. "
                    "O custo do crédito está alto demais para a capacidade atual de pagamento. Considere "
                    "renegociar a dívida, reduzir despesas ou aumentar temporariamente o valor destinado "
                    "à quitação."
                )
                r["tipo_alerta"] = "A — custo elevado"
            else:  # Moderada ou Baixa
                r["alerta"] = (
                    f"O pagamento recomendado cobre apenas {cobertura:.0f}% dos juros estimados do mês. "
                    "A taxa não é necessariamente alta, mas o saldo da dívida é muito grande em relação "
                    "à sua renda. O problema é de escala: será necessário renegociar, reduzir o saldo ou "
                    "aumentar a capacidade de pagamento."
                )
                r["tipo_alerta"] = "B — escala da dívida"

    return r


## Reproduzindo os dois casos que motivaram a distinção

In [ ]:
print("=== Caso A esperado: taxa alta (272% a.a. mediana do grupo 'custo') ===")
r = calcular_v22(renda_liquida=8000, taxa=2.5, unidade_taxa="aa", total_divida=60000)
print(f"Classificação: {r['classificacao']} | Cobertura de juros: {r.get('cobertura_juros_pct')}%")
print(r.get("alerta"))
print(f"Tipo: {r.get('tipo_alerta')}")

print()
print("=== Caso B esperado: taxa baixa (9% a.a.), dívida 14x a renda ===")
r = calcular_v22(renda_liquida=8553, taxa=0.09, unidade_taxa="aa", total_divida=120793)
print(f"Classificação: {r['classificacao']} | Cobertura de juros: {r.get('cobertura_juros_pct')}%")
print(r.get("alerta"))
print(f"Tipo: {r.get('tipo_alerta')}")

print()
print("=== Caso saudável, pra conferir que a cobertura aparece mesmo sem alerta ===")
r = calcular_v22(renda_liquida=6000, taxa=1.066, unidade_taxa="aa", total_divida=12000)
print(f"Classificação: {r['classificacao']} | Cobertura de juros: {r.get('cobertura_juros_pct')}%")
print(f"Alerta: {r.get('alerta')}")


## Gerador de resposta rápida — v2.2 (versão final)

In [ ]:
def gerar_resposta_rapida_v22(renda, taxa, unidade_taxa="aa", total_divida=None, parcela_minima=None):
    """Versão final do gerador de resposta -- usa o modelo v2.2, com os dois alertas distintos."""
    r = calcular_v22(renda, taxa, unidade_taxa, total_divida, parcela_minima)

    if "erro" in r:
        return f"⚠️ {r['erro']}"

    if r.get("alerta"):
        return f"⚠️ {r['alerta']}"

    txt = (
        f"Com renda líquida de R$ {renda:,.0f} e essa dívida a {r['taxa_anual_divida_pct']}% a.a. "
        f"({r['classificacao']}, spread de {r['spread_pp']:.0f} p.p. sobre o CDI líquido), "
        f"o modelo recomenda comprometer entre R$ {r['pagamento_recomendado'][0]:,.0f} "
        f"e R$ {r['pagamento_recomendado'][1]:,.0f} por mês, "
        f"preservando R$ {r['minimo_existencial']:,.0f} como mínimo pra viver."
    )
    if "prazo_meses" in r and not isinstance(r["prazo_meses"][0], str):
        pmin, pmax = r["prazo_meses"]
        txt += f" Nesse ritmo, a quitação leva entre {pmin:.0f} e {pmax:.0f} meses."
    return txt


print(gerar_resposta_rapida_v22(renda=3500, taxa=1.812, unidade_taxa="aa", total_divida=5000))


---
## v2.2 congelada. Próximo passo: conteúdo, não mais matemática.

O ângulo editorial que emergiu do processo inteiro é mais forte do que "calculei quanto você deve pagar por mês":

> **O modelo não olha apenas para quanto você deve ou para os juros. Ele separa três perguntas: quanto você pode pagar, quão cara é a dívida, e se o pagamento recomendado realmente consegue reduzir o saldo.**

Isso já é a espinha dorsal natural do thread pro X, do artigo do LinkedIn e do texto do Medium — a "descoberta" de que dívida cara vs. dívida grande são dois problemas diferentes, e que muita calculadora por aí trata como se fossem o mesmo.